# Getting Started with the AG2 Agent Template

This notebook demonstrates the AG2 (formerly AutoGen) agent template from Agent Starter Pack.

[AG2](https://ag2.ai) is an open-source multi-agent framework with 500K+ monthly PyPI downloads and 4,300+ GitHub stars. It enables building multi-agent systems where agents collaborate, use tools, and solve complex tasks.

## What you'll learn
- How the AG2 template works
- How to customize tools and system prompts
- How to add multiple agents with GroupChat

In [ ]:
%pip install "ag2[openai,gemini]>=0.11.4,<1.0" python-dotenv -q

## Basic Usage

The template's `run_agent()` function handles agent setup and execution:

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "your-key-here"  # Or use Vertex AI

from app.agent import run_agent

response = run_agent("What's the weather in San Francisco?")
print(response)

## Customizing: Adding Your Own Tools

You can extend the agent by registering additional tools. AG2 uses a decorator pattern for tool registration:

In [ ]:
from typing import Annotated
from autogen import AssistantAgent, UserProxyAgent, LLMConfig

llm_config = LLMConfig(config_list=[{
    "model": "gpt-4o-mini",
    "api_key": os.environ.get("OPENAI_API_KEY"),
    "api_type": "openai",
}])

assistant = AssistantAgent(
    name="Assistant",
    system_message="You are a helpful assistant. Use tools to answer questions. Reply TERMINATE when done.",
    llm_config=llm_config,
)

user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=5,
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    code_execution_config=False,
)

# Register a custom tool
@user_proxy.register_for_execution()
@assistant.register_for_llm(description="Calculate the square of a number")
def square(n: Annotated[int, "The number to square"]) -> int:
    return n * n

result = user_proxy.run(assistant, message="What is 42 squared?")

## Advanced: Multi-Agent GroupChat

AG2's key differentiator is native multi-agent orchestration. Here's how to set up a GroupChat with specialized agents:

In [ ]:
from autogen import GroupChat, GroupChatManager

researcher = AssistantAgent(
    name="Researcher",
    system_message="You research topics thoroughly and present findings.",
    llm_config=llm_config,
)

writer = AssistantAgent(
    name="Writer",
    system_message="You write clear, concise content based on research. Reply TERMINATE when done.",
    llm_config=llm_config,
)

coordinator = UserProxyAgent(
    name="Coordinator",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,
    code_execution_config=False,
)

group_chat = GroupChat(
    agents=[coordinator, researcher, writer],
    messages=[],
    max_round=6,
    speaker_selection_method="auto",
)

manager = GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config,
)

result = coordinator.run(
    manager,
    message="Write a brief overview of how AI is used in weather forecasting.",
)

## Next Steps

- Add more tools in `app/agent.py` for your specific use case
- Deploy to Cloud Run using `make deploy`
- Explore [AG2 Documentation](https://docs.ag2.ai) for advanced patterns
- See [AG2 GroupChat Tutorial](https://docs.ag2.ai/docs/tutorial/conversation-patterns) for complex workflows